In [1]:
# data

from data.get import get_data
from data.splits import load_splits

In [2]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR

from spdnet.spd import SPDTransform
from eegspdnet.nn import ChSpecConv, SCMPool, AltLogEig, AltReEig
from eegspdnet.optim import AltStiefelMetaOptimizer


In [3]:
defaults = {
    'channels_to_pick': sorted(['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'C4', 'CP5', 'CP1', 'CP2', 'CP6', 'FC3', 'FCz',
                                'FC4', 'C5', 'C1', 'C2', 'C6', 'CP3', 'CPz', 'CP4', 'FFC5h', 'FFC3h', 'FFC4h',
                                'FFC6h', 'FCC5h', 'FCC3h', 'FCC4h', 'FCC6h', 'CCP5h', 'CCP3h', 'CCP4h', 'CCP6h',
                                'CPP5h', 'CPP3h', 'CPP4h', 'CPP6h', 'FFC1h', 'FFC2h', 'FCC1h', 'FCC2h', 'CCP1h',
                                'CCP2h', 'CPP1h', 'CPP2h']),
    'low_cut_hz': 4,
    'hi_cut_hz': 124,
    's_freq': 250,
    'max_abs_val': 800,
    'trial_stop_offset_samples': 0,
    'trial_start_offset_samples': 125,
    'n_jobs': 1,
    'validation_split_point': 0.8,
}

params = {
    'batch_size': 216,
    'shuffle': True,
    'drop_last': True
}

In [4]:
mode='valid'
sub=1
name='Schirrmeister2017'

trainset, testset = load_splits(*get_data(sub, name=name, **defaults), mode=mode, **params)

Creating RawArray with float64 data, n_channels=128, n_times=1225545
    Range : 0 ... 1225544 =      0.000 ...  2451.088 secs
Ready.
Creating RawArray with float64 data, n_channels=128, n_times=616535
    Range : 0 ... 616534 =      0.000 ...  1233.068 secs
Ready.
320 events found
Event IDs: [1 2 3 4]
160 events found
Event IDs: [1 2 3 4]
Used Annotations descriptions: ['feet', 'left_hand', 'rest', 'right_hand']
Adding metadata with 4 columns
Replacing existing metadata with 4 columns
320 matching events found
No baseline correction applied
0 projection items activated
Loading data for 320 events and 875 original time points ...
0 bad epochs dropped
Loading data for 320 events and 875 original time points ...
Used Annotations descriptions: ['feet', 'left_hand', 'rest', 'right_hand']
Adding metadata with 4 columns
Replacing existing metadata with 4 columns
160 matching events found
No baseline correction applied
0 projection items activated
Loading data for 160 events and 875 original 

In [5]:
n_batches = trainset.dataset.features.shape[0] // params['batch_size']
n_epochs = 50
T_max = n_batches * n_epochs  # update scheduler after each batch

In [6]:
n_bands = 4
n_classes = 4
n_ch = len(defaults['channels_to_pick'])

In [7]:
class EEGSPDNet(nn.Module):
    
    def __init__(self, n_ch, n_bands, n_classes):
        super(__class__, self).__init__()
    
        self.conv = ChSpecConv(n_ch=n_ch, n_filters=n_bands, n_time_kernel=25)
        self.scm = SCMPool(n_bands, add_d=False)
        
        bimap_sizes = [(n_bands * n_ch) // 2**i for i in range(4)]  # half dim size with each bimap
        self.bimap1 = SPDTransform(bimap_sizes[0], bimap_sizes[1])
        self.reeig1 = AltReEig()
        self.bimap2 = SPDTransform(bimap_sizes[1], bimap_sizes[2])
        self.reeig2 = AltReEig()
        self.bimap3 = SPDTransform(bimap_sizes[2], bimap_sizes[3])
        self.reeig3 = AltReEig()
        self.log = AltLogEig(bimap_sizes[-1])
        
        vectorised_size = int(bimap_sizes[-1] * (bimap_sizes[-1] + 1) * 0.5)
        
        self.linear = nn.Linear(vectorised_size, n_classes)
        
    def forward(self, x):
        
        x = self.conv(x)
        x = self.scm(x)
        x = self.bimap1(x)
        x = self.reeig1(x)
        x = self.bimap2(x)
        x = self.reeig2(x)
        x = self.bimap3(x)
        x = self.reeig3(x)
        x = self.log(x)
        x = self.linear(x.flatten(start_dim=1))
        return x
    
model = EEGSPDNet(n_ch=n_ch, n_bands=n_bands, n_classes=n_classes)
_optim = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(_optim, T_max=T_max, eta_min=0)
optim = AltStiefelMetaOptimizer(_optim)
criterion = nn.CrossEntropyLoss()

In [8]:


# train

for epoch in range(n_epochs):
    
    model.train()
    
    train_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, batch in enumerate(trainset):
        data = batch['data']
        labels = batch['label'].squeeze().long()
        
        optim.zero_grad()
        outputs = model(data)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()
        scheduler.step()
        
        train_loss += loss.data.item()
        _, preds = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += preds.eq(labels.data).cpu().sum().data.item()
        
    final_loss = train_loss / (batch_idx + 1)
    acc = 100 * (correct / total)
    
    print(f'{epoch}: {final_loss:.2f}, {acc:.2f}%')

0: 1.39, 29.63%
1: 1.36, 43.52%
2: 1.35, 31.48%
3: 1.33, 46.76%
4: 1.32, 56.94%
5: 1.29, 69.91%
6: 1.29, 62.96%
7: 1.27, 59.72%
8: 1.26, 54.17%
9: 1.25, 50.46%
10: 1.24, 54.63%
11: 1.23, 60.65%
12: 1.22, 63.43%
13: 1.21, 66.20%
14: 1.19, 69.91%
15: 1.19, 68.06%
16: 1.18, 70.83%
17: 1.17, 70.37%
18: 1.16, 73.15%
19: 1.15, 71.30%
20: 1.15, 68.98%
21: 1.14, 71.30%
22: 1.13, 70.37%
23: 1.13, 72.69%
24: 1.13, 73.15%
25: 1.11, 72.22%
26: 1.11, 71.76%
27: 1.10, 73.61%
28: 1.09, 74.07%
29: 1.10, 74.07%
30: 1.09, 72.69%
31: 1.08, 72.22%
32: 1.08, 74.07%
33: 1.07, 74.54%
34: 1.07, 73.61%
35: 1.06, 73.61%
36: 1.06, 75.46%
37: 1.06, 75.93%
38: 1.06, 76.39%
39: 1.05, 76.39%
40: 1.05, 75.93%
41: 1.05, 74.54%
42: 1.05, 74.07%
43: 1.05, 73.61%
44: 1.06, 72.22%
45: 1.05, 71.76%
46: 1.05, 74.07%
47: 1.05, 73.15%
48: 1.05, 74.54%
49: 1.03, 76.39%


In [9]:
# test

model.eval()

test_loss = 0
correct = 0
total = 0

for batch_idx, batch in enumerate(testset):
    data = batch['data']
    labels = batch['label'].squeeze().long()
    
    outputs = model(data)
    
    loss = criterion(outputs, labels)
    
    test_loss += loss.data.item()
    _, preds = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += preds.eq(labels.data).cpu().sum().data.item()
    
final_loss = train_loss / (batch_idx + 1)
acc = 100 * (correct / total)

print(final_loss, acc)

1.033513069152832 71.875
